# Historical Climate — Monthly & Yearly Aggregation

Processes observed (historical) station climate data — precipitation, Tmax,
Tmin — into the monthly and yearly products the rest of the pipeline
(`2_future_climate_data.ipynb`, `3_future_climate_annual.ipynb`) depends on,
plus a double-mass-curve station-consistency check and annual trend plots.

**Inputs** (`All_DATA/Climate/Historical/`):
- `filled_pcp_historical_data.csv` — daily precipitation, gap-filled, per station
- `tmp_west_rapti_max.csv` / `tmp_west_rapti_min.csv` — daily Tmax / Tmin per station

**Outputs** — written directly into the folder layout already in use on disk:
- `Historical/Monthly/` — monthly totals and Jan–Dec climatology (precip, Tmax, Tmin)
- `Historical/Annual/` — yearly aggregates, summary stats, cumulative rainfall, plots
- `Historical/DMC curves/` — combined double-mass-curve plot

Aggregation logic (monthly total / yearly total / monthly climatology /
summary stats) lives in `climate_aggregation.py`

In [ ]:
import os

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator
%matplotlib inline

from climate_aggregation import (
    monthly_aggregate,
    yearly_aggregate,
    monthly_climatology,
    summary_stats,
)


In [ ]:
# --- Config: paths -----------------------------------------------------
HIST_DIR = "../All_DATA/Climate/Historical"
ANNUAL_DIR = f"{HIST_DIR}/Annual"
MONTHLY_DIR = f"{HIST_DIR}/Monthly"
DMC_DIR = f"{HIST_DIR}/DMC curves"

for _dir in (ANNUAL_DIR, MONTHLY_DIR, DMC_DIR):
    os.makedirs(_dir, exist_ok=True)

PCP_INPUT = f"{HIST_DIR}/filled_pcp_historical_data.csv"
TMAX_INPUT = f"{HIST_DIR}/tmp_west_rapti_max.csv"
TMIN_INPUT = f"{HIST_DIR}/tmp_west_rapti_min.csv"


## Precipitation — monthly & yearly aggregation

In [ ]:
pcp_data = pd.read_csv(PCP_INPUT)
pcp_stations = [c for c in pcp_data.columns if c not in ("Date", "Month", "Day", "month/day")]
pcp_data.head()


In [ ]:
monthly_pcp = monthly_aggregate(pcp_data, "Date", pcp_stations, how="sum")
monthly_pcp.to_csv(f"{MONTHLY_DIR}/historical_monthly_pcp.csv", index=False)

monthly_avg_pcp = monthly_climatology(pcp_data, "Date", pcp_stations, how="sum")
monthly_avg_pcp.to_csv(f"{MONTHLY_DIR}/historical_monthly_average_pcp.csv", index=False)

yearly_pcp = yearly_aggregate(pcp_data, "Date", pcp_stations, how="sum")
yearly_pcp.to_csv(f"{ANNUAL_DIR}/historical_yearly_pcp.csv", index=False)

yearly_stats_pcp = summary_stats(yearly_pcp, pcp_stations, transpose=True, add_sum=True)
yearly_stats_pcp.to_csv(f"{ANNUAL_DIR}/historical_yearly_stats_pcp.csv")

yearly_pcp.head()


## Precipitation — cumulative rainfall & double-mass curve

The double-mass curve (cumulative station rainfall vs. cumulative
basin-average rainfall) is a standard QA check: a station whose line bends
away from the shared trend of the others may have an inhomogeneity (gauge
move, instrument change, etc.) worth investigating.

In [ ]:
cumulative_pcp = yearly_pcp.copy()
cumulative_pcp["Average"] = cumulative_pcp[pcp_stations].mean(axis=1).round(2)
cumulative_pcp[pcp_stations + ["Average"]] = cumulative_pcp[pcp_stations + ["Average"]].cumsum(axis=0)
cumulative_pcp.to_csv(f"{ANNUAL_DIR}/historical_yearly_cumulative_pcp.csv", index=False)
cumulative_pcp.head()


In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

x = cumulative_pcp["Average"]
colors = plt.cm.tab20.colors

for i, station in enumerate(pcp_stations):
    ax.plot(
        x, cumulative_pcp[station],
        label=station, color=colors[i % len(colors)],
        linewidth=2, marker="o", markersize=3, alpha=0.9,
    )

ax.set_title("Double Mass Curve of Rainfall Stations", fontsize=18, fontweight="bold", pad=15)
ax.set_xlabel("Cumulative Average Annual Rainfall of All Stations (mm)", fontsize=13, labelpad=10)
ax.set_ylabel("Cumulative Annual Rainfall at Individual Stations (mm)", fontsize=13, labelpad=10)

ax.grid(which="major", linestyle="--", linewidth=0.8, alpha=0.5)
ax.grid(which="minor", linestyle=":", linewidth=0.5, alpha=0.3)
ax.xaxis.set_minor_locator(AutoMinorLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_linewidth(1.2)
ax.spines["bottom"].set_linewidth(1.2)
ax.tick_params(axis="both", labelsize=11)

legend = ax.legend(
    title="Stations", title_fontsize=11, fontsize=9,
    loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=True, ncol=1,
)
legend.get_frame().set_edgecolor("gray")
legend.get_frame().set_linewidth(0.8)

plt.tight_layout()
fig.savefig(f"{DMC_DIR}/Combined_Double_Mass_Curve.png", dpi=600, bbox_inches="tight")
plt.show()


## Precipitation — annual time series

In [ ]:
def plot_annual_timeseries(df, station_cols, title, ylabel, out_path,
                           year_col="Year", year_range=None, colors=None,
                           show_average=False):
    """Multi-station annual line plot, optionally restricted to `year_range`
    (a (start, end) tuple) with a black dashed cross-station average line."""
    plot_df = df.copy()
    if year_range is not None:
        start, end = year_range
        plot_df = plot_df[(plot_df[year_col] >= start) & (plot_df[year_col] <= end)]

    fig, ax = plt.subplots(figsize=(14, 8))
    colors = colors if colors is not None else plt.cm.tab20.colors

    for i, station in enumerate(station_cols):
        ax.plot(
            plot_df[year_col], plot_df[station],
            label=f"Station {station}", color=colors[i % len(colors)],
            linewidth=1.8, alpha=0.85,
        )

    if show_average:
        average = plot_df[station_cols].mean(axis=1)
        ax.plot(plot_df[year_col], average, color="black", linewidth=3,
                linestyle="--", label="Average")

    ax.set_title(title, fontsize=18, fontweight="bold", pad=15)
    ax.set_xlabel("Year", fontsize=13)
    ax.set_ylabel(ylabel, fontsize=13)

    ax.grid(which="major", linestyle="--", linewidth=0.8, alpha=0.5)
    ax.grid(which="minor", linestyle=":", linewidth=0.5, alpha=0.3)
    ax.xaxis.set_minor_locator(AutoMinorLocator())
    ax.yaxis.set_minor_locator(AutoMinorLocator())
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(labelsize=11)

    legend = ax.legend(title="Stations", loc="center left", bbox_to_anchor=(1.02, 0.5),
                        fontsize=9, title_fontsize=10, frameon=True)
    legend.get_frame().set_edgecolor("gray")

    plt.tight_layout()
    fig.savefig(out_path, dpi=600, bbox_inches="tight")
    plt.show()


In [ ]:
plot_annual_timeseries(
    yearly_pcp, pcp_stations,
    title="Annual Rainfall Time Series",
    ylabel="Annual Rainfall (mm)",
    out_path=f"{ANNUAL_DIR}/Annual_Rainfall_Timeseries.png",
)


## Temperature (Tmax / Tmin) — yearly aggregation & climatology

In [ ]:
tmax_data = pd.read_csv(TMAX_INPUT)
tmax_stations = [c for c in tmax_data.columns if c != "Date"]

yearly_tmax = yearly_aggregate(tmax_data, "Date", tmax_stations, how="mean")
yearly_tmax[tmax_stations] = yearly_tmax[tmax_stations].round(2)
yearly_tmax.to_csv(f"{ANNUAL_DIR}/tmax_historical_yearly_aggregation.csv", index=False)
summary_stats(yearly_tmax, tmax_stations).to_csv(f"{ANNUAL_DIR}/tmax_historical_yearly_stats.csv")

monthly_climatology(tmax_data, "Date", tmax_stations, how="mean").to_csv(
    f"{MONTHLY_DIR}/historical_monthly_average_Tmax.csv", index=False
)

tmin_data = pd.read_csv(TMIN_INPUT)
tmin_stations = [c for c in tmin_data.columns if c != "Date"]

yearly_tmin = yearly_aggregate(tmin_data, "Date", tmin_stations, how="mean")
yearly_tmin[tmin_stations] = yearly_tmin[tmin_stations].round(2)
yearly_tmin.to_csv(f"{ANNUAL_DIR}/tmin_historical_yearly_aggregation.csv", index=False)
summary_stats(yearly_tmin, tmin_stations).to_csv(f"{ANNUAL_DIR}/tmin_historical_yearly_stats.csv")

monthly_climatology(tmin_data, "Date", tmin_stations, how="mean").to_csv(
    f"{MONTHLY_DIR}/historical_monthly_average_Tmin.csv", index=False
)

yearly_tmax.head()


## Temperature — 1985–2015 trend plots

In [ ]:
plot_annual_timeseries(
    yearly_tmax, tmax_stations,
    title="Annual Mean Maximum Temperature (1985–2015)",
    ylabel="Annual Mean Tmax (°C)",
    out_path=f"{ANNUAL_DIR}/Annual_Tmax_1985_2015.png",
    year_range=(1985, 2015), colors=plt.cm.tab10.colors, show_average=True,
)


In [ ]:
plot_annual_timeseries(
    yearly_tmin, tmin_stations,
    title="Annual Mean Minimum Temperature (1985–2015)",
    ylabel="Annual Mean Tmin (°C)",
    out_path=f"{ANNUAL_DIR}/Annual_TMin_1985_2015.png",
    year_range=(1985, 2015), colors=plt.cm.tab10.colors, show_average=True,
)
